## shouldSplit: train + inference

Ниже короткий рабочий блок для:
1. обучения `shouldSplit` модели через существующий pipeline,
2. сохранения артефакта,
3. инференса по новым объявлениям из артефакта.

In [7]:
import avito

ModuleNotFoundError: No module named 'avito'

In [1]:
from pathlib import Path

import pandas as pd

from avito.classifier import train_should_split_models
from avito.config import AvitoCaseConfig
from avito.features import ShouldSplitFeatureConfig
from avito.inference import predict_should_split_from_artifact


case_config = AvitoCaseConfig.from_default_yaml()
feature_config = ShouldSplitFeatureConfig(
    include_extra_text_features=case_config.should_split.include_extra_text_features,
)

data_path = Path("avito/data/rnc_dataset.csv")
artifact_path = Path("checkpoints/avito_should_split_model.joblib")

train_df = pd.read_csv(data_path)
result = train_should_split_models(
    df=train_df,
    include_embeddings=False,
    feature_config=feature_config,
    training_config=case_config.should_split.training,
)

artifact_path.parent.mkdir(parents=True, exist_ok=True)
payload = {
    "best_model_name": result.model_name,
    "pipeline": result.pipeline,
    "with_embeddings": False,
    "feature_config": feature_config.model_dump(mode="json"),
}

import joblib
joblib.dump(payload, artifact_path)

sample_df = train_df.iloc[:5][["description", "sourceMcId", "sourceMcTitle"]].copy()
inference = predict_should_split_from_artifact(
    df=sample_df,
    artifact_path=artifact_path,
)

sample_df["shouldSplit_pred"] = inference.predictions
sample_df["shouldSplit_proba"] = inference.probabilities
sample_df.head()

ModuleNotFoundError: No module named 'avito'